# Plot flux map of galaxy

- Choose Energy bin
- Bin using PSF
- Load exposure map 
- Re-bin exposure map to same format
- Divide count map 

In [1]:
import ROOT, glob, os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [2]:
dirDir= "/home/users/l/lebrun/"
resultPath = dirDir + "results/"
figPath = dirDir + "figures/"
dataPath = "/dpnc/beegfs/dampe/tryf/ForKimberly/filtered_cos_theta_bins/"
exposurePath = "/dpnc/beegfs/dampe/tryf/ForKimberly/filtered_cos_theta_bins/exposureMaps/"
resPath = dirDir + "results/"
exponent = 0

In [4]:
Energy = 2.818 # GeV
Emin = 1
Emax_HET = 1e4
n_E_bins_EffArea = 40
n_E_bins_PSF = 8

E_bins_edge_EffArea = np.logspace(np.log10(Emin), np.log10(Emax_HET), n_E_bins_EffArea + 1)
E_bins_center_EffArea = np.sqrt(E_bins_edge_EffArea[:-1] * E_bins_edge_EffArea[1:])

print(E_bins_center_EffArea)

E_bins_edge_PSF = np.logspace(np.log10(Emin), np.log10(Emax_HET), n_E_bins_PSF + 1)
E_bins_center_PSF = np.sqrt(E_bins_edge_PSF[:-1] * E_bins_edge_PSF[1:])

[1.12201845e+00 1.41253754e+00 1.77827941e+00 2.23872114e+00
 2.81838293e+00 3.54813389e+00 4.46683592e+00 5.62341325e+00
 7.07945784e+00 8.91250938e+00 1.12201845e+01 1.41253754e+01
 1.77827941e+01 2.23872114e+01 2.81838293e+01 3.54813389e+01
 4.46683592e+01 5.62341325e+01 7.07945784e+01 8.91250938e+01
 1.12201845e+02 1.41253754e+02 1.77827941e+02 2.23872114e+02
 2.81838293e+02 3.54813389e+02 4.46683592e+02 5.62341325e+02
 7.07945784e+02 8.91250938e+02 1.12201845e+03 1.41253754e+03
 1.77827941e+03 2.23872114e+03 2.81838293e+03 3.54813389e+03
 4.46683592e+03 5.62341325e+03 7.07945784e+03 8.91250938e+03]


### Identify energy bin

In [ ]:
# Find the bin index such that E_bins_edge_HET[bin_idx] <= Energy < E_bins_edge_HET[bin_idx+1]
bin_idx_EffArea = np.searchsorted(E_bins_center_EffArea, Energy, side='right') - 1
bin_idx_PSF = np.searchsorted(E_bins_center_PSF, Energy, side='right') - 1

energy_exposure_map = E_bins_center_EffArea[bin_idx_EffArea]
energy_exposure_map_str = f"{energy_exposure_map:.3f}GeV" 

print(f"Energy {Energy} GeV falls in EffArea bin index: {bin_idx_EffArea} (edges: {E_bins_edge_EffArea[bin_idx_EffArea]}, {E_bins_edge_EffArea[bin_idx_EffArea+1]})")
print(f"Energy {Energy} GeV falls in PSF bin index: {bin_idx_PSF} (edges: {E_bins_edge_PSF[bin_idx_PSF]}, {E_bins_edge_PSF[bin_idx_PSF+1]})")

Energy 2.818 GeV falls in EffArea bin index: 3 (edges: 1.99526231496888, 2.51188643150958)
Energy 2.818 GeV falls in PSF bin index: 0 (edges: 1.0, 3.162277660168379)


### Load PSF

In [6]:
resolution_df = pd.read_csv(resPath + 'resolution_per_bin_weighted_exp0.0.csv')
print(resolution_df.head())
print(resolution_df[resolution_df['E_bin']== bin_idx_PSF])
psf = resolution_df[resolution_df['E_bin'] == bin_idx_PSF]['angular_resolution_w'].max()
print(f"PSF (max angular_resolution_w) for E_bin {bin_idx_PSF}: {psf}")

   E_bin  theta_bin  angular_resolution_w
0      0          0              0.679101
1      0          1              0.529916
2      0          2              0.491009
3      0          3              0.433219
4      0          4              0.376656
   E_bin  theta_bin  angular_resolution_w
0      0          0              0.679101
1      0          1              0.529916
2      0          2              0.491009
3      0          3              0.433219
4      0          4              0.376656
PSF (max angular_resolution_w) for E_bin 0: 0.6791005161181329


### Load exposure map